# Typography and density

> Independent text, spacing, and radius scales, base element styles, surface relief tokens, prose typography, and optional IBM Plex fonts.

AnswerUI controls text size, spacing, and corner radius with three independent CSS custom properties: `--text`, `--space`, and `--radius`. Override any of them on `:root` for the whole page or on a `[ui]` container for one region. You can tighten `--space` in a dialog without shrinking `--text`. You can also enlarge `--text` in a reading pane without changing padding or corner radius.

In [ ]:
from answerui.preview import preview

## Scales and tokens

Steps with negative suffixes are smaller than the base. Steps with positive suffixes are larger. The stylesheet uses spacing tokens for internal control padding and gaps between layout items.

- Text (`--text: 1rem`): `--text-1` (`0.875`), `--text1` (`1.25`), `--text2` (`1.5`), `--text3` (`2`).
- Space (`--space: 1rem`): `--space-3` (`0.25`), `--space-2` (`0.5`), `--space-1` (`0.75`), `--space1` (`1.5`), `--space2` (`2`), `--space3` (`3`).
- Radius (`--radius: 0.5rem`): `--radius-1` (`0.5`), `--radius1` (`2`), plus the fixed pill radius `--radius-full` (`9999px`).

The token layer also defines font weights (`--weight: 400`, `--weight1: 600`), line heights (`--leading: 1.5`, `--leading-1: 1.25`), motion timings (`--motion: 150ms`, `--ease: cubic-bezier(.2, 0, 0, 1)`), and `@keyframes ui-spin`. Under `prefers-reduced-motion: reduce`, `--motion` becomes `0ms`.

Base tokens inherit to descendants. The stylesheet declares every derived step on all elements and pseudo-elements inside `:root` and `[ui]`. Overriding a base on a container recalculates every derived step inside that container. Set a base token to scale a whole region. Override a single derived step only for a one-off exception on one element, because derived steps are still computed from the base on every descendant.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer tokens {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope):root {
      --font: system-ui, sans-serif;
      --font-mono: ui-monospace, Menlo, Consolas, monospace;
      --text: 1rem;
      --weight: 400; --weight1: 600;
      --leading: 1.5; --leading-1: 1.25;
      --prose-scale: .75;
      --prose-font-scale: 1;
      --prose-leading: 1.45;
      --prose-measure: 65ch;
      --space: 1rem;
      --radius: .5rem; --radius-full: 9999px;
      --relief: 1;
      --motion: 150ms;
      --ease: cubic-bezier(.2, 0, 0, 1);
      @media (prefers-reduced-motion: reduce) { --motion: 0ms; }
    }

    :where(:scope, *), ::before, ::after {
      --text-1: calc(var(--text) * .875);
      --text1: calc(var(--text) * 1.25);
      --text2: calc(var(--text) * 1.5);
      --text3: calc(var(--text) * 2);
      --space-3: calc(var(--space) * .25);
      --space-2: calc(var(--space) * .5);
      --space-1: calc(var(--space) * .75);
      --space1: calc(var(--space) * 1.5);
      --space2: calc(var(--space) * 2);
      --space3: calc(var(--space) * 3);
      --radius-1: calc(var(--radius) * .5);
      --radius1: calc(var(--radius) * 2);
    }

    @keyframes ui-spin { to { rotate: 1turn; } }
  }
}
'''

## Base styles and scoping

Inside `:root` and `[ui]`, the reset sets `box-sizing: border-box` and clears default margins on block elements (`body`, headings, `p`, lists, `blockquote`, `pre`, `figure`, `figcaption`, and `hr`). Form controls inherit `font`. Media elements (`img`, `picture`, `video`, `canvas`, and `svg`) use `max-inline-size: 100%` and `block-size: auto`, except height-only images (`img[height]:not([width])`), which keep their HTML `height` attribute with `inline-size: auto` and `object-fit: contain`. With block margins cleared, layout attributes such as `[stack]` and `[grid]` control spacing. The reset does not remove native browser focus outlines.

The base layer sets `--color-background`, `--color-text`, and `--color-accent` on `:root` and `[ui]`. It sets `--weight`, `--text`, `--leading`, and `--font` on `body` and `[ui]`. Headings use `--font-heading` (defaulting to `inherit`), `--weight1`, `--leading-1`, and `text-wrap: balance`. Heading sizes use `--text3` for `h1`, `--text2` for `h2`, `--text1` for `h3`, and `--text` for `h4` through `h6`. Links use `--color-accent` with `text-underline-offset: .15em`. Code elements (`code`, `kbd`, `pre`, `samp`) use `--font-mono`. `small` uses `--text-1`. Icons matching `svg.ui-icon`, `svg[icon]`, or `svg:has(> use)` are sized to `1.25em` square. Keyboard-focused elements (`:focus-visible`) show `--ring` with a `2px` offset. Use `.ui-divider` for a `1px` top border in `--color-border`. Unmarked tables and form controls remain unstyled native elements.

Add `[ui]` to any container to override base scales, colour scheme, or hue locally without loading a second stylesheet. Wrap an opaque widget and its popup host in `[unstyle]` to stop AnswerUI selectors from matching inside it. Ordinary CSS inheritance still passes through `[unstyle]`. Add `[ui]` to an element inside `[unstyle]` to start a new AnswerUI scope.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer reset {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *), ::before, ::after { box-sizing: border-box; }
    :where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, pre, figure, figcaption, hr) { margin: 0; }
    :where(button, input, textarea, select) { font: inherit; }
    :where(img, picture, video, canvas, svg) { max-inline-size: 100%; }
    :where(img:not([height]), img[width], picture, video, canvas, svg) { block-size: auto; }
    :where(img[height]:not([width])) { inline-size: auto; object-fit: contain; object-position: left top; }
  }
}
@layer base {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope):is([ui], :root) { background: var(--color-background); color: var(--color-text); accent-color: var(--color-accent); }
    :where(:scope, *):is([ui], body) { font: var(--weight) var(--text)/var(--leading) var(--font); }
    :where(:scope, *):focus-visible { outline: var(--ring); outline-offset: 2px; }
    :where(h1, h2, h3, h4, h5, h6) { font-family: var(--font-heading, inherit); font-weight: var(--weight1); line-height: var(--leading-1); text-wrap: balance; }
    :where(h1) { font-size: var(--text3); }
    :where(h2) { font-size: var(--text2); }
    :where(h3) { font-size: var(--text1); }
    :where(h4, h5, h6) { font-size: var(--text); }
    :where(a) { color: var(--color-accent); text-underline-offset: .15em; }
    :where(code, kbd, pre, samp) { font-family: var(--font-mono); }
    :where(small) { font-size: var(--text-1); }
    :where(svg.ui-icon, svg[icon], svg:has(> use)) { inline-size: 1.25em; block-size: 1.25em; flex: none; vertical-align: -.25em; }
    .ui-divider { border: 0; border-block-start: 1px solid var(--color-border); }
  }
}
'''

## Surface relief and shadows

Raised controls use `--bevel` and `--raised`. Sunken controls such as inputs use `--bevel-in` and `--sunken`. Both bevels model top-left lighting. With `--bevel`, `--hi` is on the top and left edges and `--lo` is on the bottom and right. With `--bevel-in`, those edges are reversed.

`--hi` and `--lo` are computed by shifting the `oklch` lightness of `--face` (default `var(--color-surface)`) by `+0.16` and `-0.16`. Deriving rim colours from `--face` keeps highlights lighter and shadows darker in both light and dark schemes. Fixed palette steps cannot do that because their lightness order reverses between schemes. `--tint` is an 8% opacity tint derived from `currentColor`.

`--relief` (default `1`) scales the alpha of `--hi`, `--lo`, and `--shadow-color` without changing labels or borders. Setting `--relief: 0` hides bevels and control shadows while `--face` and borders stay visible. `--floating` uses `--floating-shadow`, which does not scale with `--relief`. Dialogs and menus still have drop shadows when `--relief` is `0`.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer tokens {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *)[ui], :where(:scope):root { --face: var(--color-surface); }
    :where(:scope, *), ::before, ::after {
      --tint: oklch(from currentColor l c h / .08);
      --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
      --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
      --shadow-color: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
      --floating-shadow: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
      --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
      --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
      --raised: 0 2px 4px var(--shadow-color);
      --sunken: inset 0 2px 4px var(--shadow-color);
      --floating: 0 4px 12px var(--floating-shadow), 0 12px 32px var(--floating-shadow);
    }
  }
}
'''

## Varying density by region

Both panels below use identical markup. The left panel uses the defaults. On the right panel, `--space` is `.75rem` and `--radius` is `.375rem`, while `--text` stays at `1rem`.

In [ ]:
def panel(density, sizes):
    return f'''
    <section ui style="{sizes}" card stack>
      <h3>{density}</h3>
      <p>Manage settings for this workspace.</p>
      <label stack class="gap-1">Display name
        <input input value="Ada Lovelace">
      </label>
      <div cluster><button type="button" btn>Save</button><a href="#help">Help</a></div>
    </section>
    '''

spacious = panel('Spacious', '')
compact = panel('Compact', '--space:.75rem;--radius:.375rem')
preview(f'<div grid>{spacious}{compact}</div>', height='17rem')

```html
<div grid>
    <section ui style="" card stack>
      <h3>Spacious</h3>
      <p>Manage settings for this workspace.</p>
      <label stack style="--gap:var(--space-3)">Display name
        <input input value="Ada Lovelace">
      </label>
      <div cluster><button type="button" btn>Save</button><a href="#help">Help</a></div>
    </section>
    
    <section ui style="--space:.75rem;--radius:.375rem" card stack>
      <h3>Compact</h3>
      <p>Manage settings for this workspace.</p>
      <label stack style="--gap:var(--space-3)">Display name
        <input input value="Ada Lovelace">
      </label>
      <div cluster><button type="button" btn>Save</button><a href="#help">Help</a></div>
    </section>
    </div>
```

## Prose typography

Add `[prose]` to a container for long-form document typography covering headings, paragraphs, lists, links, blockquotes, code, `kbd`, `mark`, tables, media, `.lead`, and `hr`.

Vertical spacing uses `--sp`, computed as `calc(var(--prose-scale) * 1em)` (default `--prose-scale: .75`). Block elements use top margins for vertical spacing. Lists immediately following a paragraph (`p + ul`, `p + ol`) use a tighter `0.25 * --sp` top margin. Headings have larger top margins (`1.4 * --sp` for `h1` and `h2`, `1.2 * --sp` for `h3`, `1.25 * --sp` for `h4` through `h6`). The stylesheet also adjusts the top margin of the element right after each heading. Code blocks (`pre`) use `line-height: 1.25` without changing `--leading` or `--prose-leading`. Direct `:first-child` and `:last-child` elements have zero outer margin. Tables inside `[prose]` scroll horizontally and wrap cell text separately from the `[table]` component.

Prose font size is `calc(var(--text) * var(--prose-font-scale))` (default `--prose-font-scale: 1`). Line height uses `--prose-leading` (default `1.45`). Maximum width uses `--prose-measure` (default `65ch`). Four `[prose-size]` presets set `--prose-scale` and `--prose-font-scale` together: `sm` (`0.7`, `0.875`), `lg` (`0.8`, `1.125`), `xl` (`0.9`, `1.25`), and `2xl` (`1.1`, `1.5`). In a notebook or narrow pane, you can set `--prose-measure: none` and lower `--prose-leading` without affecting the surrounding UI.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer base {
  @scope (:root, [ui]) to ([unstyle]) {
    /* Typography folded in from typrose, based on Tailwind Labs' typography styles.
     * MIT License
     * Copyright (c) 2026 Answer.AI
     *
     * Permission is hereby granted, free of charge, to any person obtaining a copy
     * of this software and associated documentation files (the "Software"), to deal
     * in the Software without restriction, including without limitation the rights
     * to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
     * copies of the Software, and to permit persons to whom the Software is
     * furnished to do so, subject to the following conditions:
     *
     * The above copyright notice and this permission notice shall be included in all
     * copies or substantial portions of the Software.
     *
     * THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
     * IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
     * FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
     * AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
     * LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
     * OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
     * SOFTWARE.
     */

    :where(:scope, *)[prose] {
      --sp: calc(var(--prose-scale) * 1em);
      max-inline-size: var(--prose-measure); overflow-wrap: anywhere;
      font-size: calc(var(--text) * var(--prose-font-scale)); line-height: var(--prose-leading); color: var(--color-text);

      /* Use block-start margins for vertical spacing; outer edges are cleared at the container boundary. */
      :where(p, ol, ul, dl) { margin-block-start: var(--sp); }
      :where(dt) { margin-block-start: var(--sp); font-weight: var(--weight1); }
      :where(dd) { margin-block-start: calc(var(--sp) * .25); padding-inline-start: 1.375em; }
      :where(li) { margin-block-start: calc(var(--sp) * .25); }
      :where(ol, ul) {
        padding-inline-start: 1.7em;
        > :where(li) { padding-inline-start: .375em; }
        > :where(li)::marker { color: var(--color-muted); }
        :where(ol, ul) { margin: 0; }
      }
      :where(ol) {
        list-style-type: decimal;
        > :where(li)::marker { font-weight: var(--weight); }
        &:where([type="A"], [type="A" s]) { list-style-type: upper-alpha; }
        &:where([type="a"], [type="a" s]) { list-style-type: lower-alpha; }
        &:where([type="I"], [type="I" s]) { list-style-type: upper-roman; }
        &:where([type="i"], [type="i" s]) { list-style-type: lower-roman; }
        &:where([type="1"]) { list-style-type: decimal; }
      }
      :where(ul) { list-style-type: disc; }
      > :where(ol, ul) > li {
        :where(p) { margin-block-start: calc(var(--sp) * .5); }
        > :where(p:first-child) { margin-block-start: calc(var(--sp) * .75); }
      }
      :where(p + :is(ol, ul)) { margin-block-start: calc(var(--sp) * .25); }

      :where(h1) { font-size: 2.25em; font-weight: 800; line-height: 1.1; margin-block-start: calc(var(--sp) * 1.4); :where(strong) { font-weight: 900; } }
      :where(h2) { font-size: 1.5em; font-weight: 700; line-height: 1.3; margin-block-start: calc(var(--sp) * 1.4); :where(strong) { font-weight: 800; } }
      :where(h3) { font-size: 1.25em; font-weight: var(--weight1); line-height: 1.4; margin-block-start: calc(var(--sp) * 1.2); }
      :where(h4, h5, h6) { font-size: 1em; font-weight: var(--weight1); line-height: 1.4; margin-block-start: calc(var(--sp) * 1.25); }
      :where(:is(h3, h4, h5, h6) strong) { font-weight: 700; }
      :where(.lead) { font-size: 1.25em; line-height: 1.6; color: var(--color-muted); margin-block-start: var(--sp); }

      :where(a:not([btn])) { color: var(--color-accent); text-decoration: underline; text-underline-offset: .15em; font-weight: 500; }
      :where(strong) { font-weight: var(--weight1); }

      :where(img, picture, video, figure) { margin-block-start: calc(var(--sp) * 1.5); }
      :where(picture) { display: block; }
      :where(img, video) { display: block; border-radius: var(--radius); }
      :where(figcaption) { font-size: .875em; line-height: 1.4; color: var(--color-muted); margin-block-start: calc(var(--sp) * .5); }
      :where(small) { font-size: .875em; color: var(--color-muted); }

      :where(blockquote) {
        font-style: italic; font-weight: 500; margin-block-start: calc(var(--sp) * 1.5); padding-inline-start: 1em;
        border-inline-start: .25rem solid var(--color-border); quotes: "\201C" "\201D" "\2018" "\2019";
      }

      :where(*:not(pre) > code) {
        font-size: .875em; font-weight: var(--weight1); background: var(--color-surface);
        padding: .1em .3rem; border-radius: var(--radius-1); line-height: 1.2rem;
      }
      :where(pre) {
        font-size: inherit; line-height: 1.25; white-space: pre-wrap; overflow-wrap: anywhere; overflow-x: auto;
        padding: .25rem .3rem; border-radius: var(--radius-1); background: var(--color-surface); margin-block-start: var(--sp);
        :where(code) { padding: 0; background: none; font: inherit; }
      }
      :where(kbd) {
        font-size: .875em; font-weight: 500; font-family: inherit; padding: .125em .3125em; border-radius: var(--radius-1);
        box-shadow: 0 0 0 1px var(--color-border), 0 3px 0 var(--color-border);
      }
      :where(mark) { background: var(--v-3); color: var(--v2); padding: .1em .2em; border-radius: var(--radius-1); }

      /* Prose tables wrap and scroll, independently of the application [table] skin. */
      :where(table) {
        display: block; inline-size: fit-content; max-inline-size: calc(100% + 1em); overflow-x: auto; border-collapse: collapse;
        font-size: .875em; line-height: 1.5; margin-block: calc(var(--sp) * 2) 0; margin-inline: -.5em;
        :where(caption) { inline-size: max-content; max-inline-size: 100%; padding-inline: .5em; }
        :where(th, td) { text-align: start; overflow-wrap: break-word; }
        :where(thead) {
          border-block-end: 1px solid var(--color-border);
          :where(th) { font-weight: var(--weight1); vertical-align: bottom; padding-block: 0 calc(var(--sp) * .5); padding-inline: .5em; }
        }
        :where(tbody) {
          :where(tr:not(:last-child)) { border-block-end: 1px solid var(--color-border); }
          :where(th, td) { vertical-align: baseline; padding-block: calc(var(--sp) * .5); padding-inline: .5em; }
        }
        :where(tfoot) {
          border-block-start: 1px solid var(--color-border);
          :where(th, td) { vertical-align: top; padding-block: calc(var(--sp) * .5); padding-inline: .5em; }
        }
      }

      :where(hr) {
        border: 0; margin-block-start: var(--sp); position: relative; block-size: 1.25rem;
        background: radial-gradient(circle, transparent 6px, var(--color-border) 6px 8px, transparent 8px) center / 20px 20px no-repeat;
        &::before, &::after { content: ''; position: absolute; inset-block-start: 50%; inline-size: calc(50% - .75rem); border-block-end: 1px solid var(--color-border); }
        &::before { inset-inline-start: 0; }
        &::after { inset-inline-end: 0; }
      }

      :where(h1 + *) { margin-block-start: calc(var(--sp) * 1.5); }
      :where(h2 + *) { margin-block-start: var(--sp); }
      :where(:is(h3, h4, h5, h6) + *) { margin-block-start: calc(var(--sp) * .5); }
      :where(picture > img, figure > :not(figcaption)) { margin-block: 0; }
      > :first-child { margin-block-start: 0; }
      > :last-child { margin-block-end: 0; }
    }

    [prose-size=sm] { --prose-scale: .7; --prose-font-scale: .875; }
    [prose-size=lg] { --prose-scale: .8; --prose-font-scale: 1.125; }
    [prose-size=xl] { --prose-scale: .9; --prose-font-scale: 1.25; }
    [prose-size="2xl"] { --prose-scale: 1.1; --prose-font-scale: 1.5; }
  }
}
'''

In [ ]:
article = '''<h2>Notes in a narrow pane</h2>
<p>Override prose variables on a container to adjust reading width and line spacing:</p>
<ul><li>Limit line width with <code>--prose-measure</code>.</li><li>Set <code>--prose-measure: none</code> for full-width panes.</li></ul>
<pre><code>parts = walk(workspace)
print(len(parts))</code></pre>
<p>The last block has no bottom margin.</p>'''
preview(f'''<div grid style="--grid-min:17rem">
  <article card prose style="--prose-measure:28ch">{article}</article>
  <article card prose style="--text:.875rem;--prose-leading:1.3;--prose-measure:none">{article}</article>
</div>''', height='23rem')

```html
<div grid style="--grid-min:17rem">
  <article card prose style="--prose-measure:28ch"><h2>Notes in a narrow pane</h2>
<p>Override prose variables on a container to adjust reading width and line spacing:</p>
<ul><li>Limit line width with <code>--prose-measure</code>.</li><li>Set <code>--prose-measure: none</code> for full-width panes.</li></ul>
<pre><code>parts = walk(workspace)
print(len(parts))</code></pre>
<p>The last block has no bottom margin.</p></article>
  <article card prose style="--text:.875rem;--prose-leading:1.3;--prose-measure:none"><h2>Notes in a narrow pane</h2>
<p>Override prose variables on a container to adjust reading width and line spacing:</p>
<ul><li>Limit line width with <code>--prose-measure</code>.</li><li>Set <code>--prose-measure: none</code> for full-width panes.</li></ul>
<pre><code>parts = walk(workspace)
print(len(parts))</code></pre>
<p>The last block has no bottom margin.</p></article>
</div>
```

## Optional fonts

By default, `--font` uses `system-ui, sans-serif` and `--font-mono` uses `ui-monospace, Menlo, Consolas, monospace`. To use IBM Plex, load `fonts.css` and set `--font: 'IBM Plex Sans', system-ui, sans-serif` and `--font-mono: 'IBM Plex Mono', ui-monospace, monospace`. `fonts.css` declares `IBM Plex Sans` (400 normal, 400 italic, 600 normal), `IBM Plex Serif` (600 normal), and `IBM Plex Mono` (400 normal). Set `--font-heading` (for example, `'IBM Plex Serif', serif`) on `:root` or a `[ui]` region to use Plex Serif for headings. Layout and spacing scales do not depend on `fonts.css`.

In [ ]:
#| hide
#| asset fonts.css
_fonts = r'''
@font-face { font-family: 'IBM Plex Sans'; font-weight: 400; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-400-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Sans'; font-weight: 400; font-style: italic; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-400-italic.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Sans'; font-weight: 600; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-600-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Serif'; font-weight: 600; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-serif-latin-600-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Mono'; font-weight: 400; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-mono-latin-400-normal.woff2') format('woff2'); }
'''
